In [ ]:
from collections.abc import Callable, Iterable
from enum import Enum
from functools import partial
from typing import Any, NamedTuple

import matplotlib.pyplot as plt
import numpy as np
import scipy.sparse
import tqdm.notebook

from gacha_model import COND_PROB_5_STAR, COND_PROB_6_STAR, PityModel
from plot_tools import draw_multi_cdf_fig, draw_multi_pmf_cdf_fig, draw_pmf_cdf_fig
from random_variable import FiniteDist

In [ ]:
# 在联动寻访中获得 UP 6★ 干员和全部 2 名 UP 5★ 干员各至少若干次所需寻访次数的分布

class 状态类(NamedTuple):
    六星水位: int
    五星水位: int
    已获取UP六星干员数量: int
    已获取UP五星干员乙数量: int
    已获取UP五星干员丙数量: int


def 获取状态(*, 六星水位: int, 五星水位: int, 已获取UP六星干员数量: int, 已获取UP五星干员乙数量: int, 已获取UP五星干员丙数量: int) -> 状态类:
    return 状态类(
        六星水位=六星水位,
        五星水位=五星水位,
        已获取UP六星干员数量=min(已获取UP六星干员数量, 已获取UP六星干员数量上限),
        已获取UP五星干员乙数量=min(已获取UP五星干员乙数量, 已获取UP五星干员乙数量上限),
        已获取UP五星干员丙数量=min(已获取UP五星干员丙数量, 已获取UP五星干员丙数量上限),
    )


def 状态转移(起始状态: 状态类, *, 是第10抽: bool, 是第120抽: bool) -> list[tuple[状态类, float]]:
    assert not (是第10抽 and 是第120抽), "不能同时是第10抽和第120抽"

    转移概率列表: list[tuple[状态类, float]] = []

    起始六星水位 = 起始状态.六星水位
    起始五星水位 = 起始状态.五星水位
    起始已获取UP六星干员数量 = 起始状态.已获取UP六星干员数量
    起始已获取UP五星干员乙数量 = 起始状态.已获取UP五星干员乙数量
    起始已获取UP五星干员丙数量 = 起始状态.已获取UP五星干员丙数量

    if 是第120抽 and 起始已获取UP六星干员数量 == 0:  # 第120抽时，若未获取过UP六星干员，则必定获取
        转移概率列表.append((获取状态(六星水位=0, 五星水位=0, 已获取UP六星干员数量=1, 已获取UP五星干员乙数量=起始已获取UP五星干员乙数量, 已获取UP五星干员丙数量=起始已获取UP五星干员丙数量), 1))

    else:
        # 计算抽到不同星级干员的概率
        六星概率 = COND_PROB_6_STAR[起始六星水位]
        if 是第10抽 and 起始五星水位 == 9:
            五星概率 = 1 - 六星概率
        else:
            五星概率 = np.clip(COND_PROB_5_STAR[起始五星水位], 0, 1 - 六星概率)
        四星或三星概率 = 1 - 六星概率 - 五星概率

        if 起始已获取UP五星干员乙数量 == 0 and 起始已获取UP五星干员丙数量 > 0:
            乙概率 = 五星概率
            丙概率 = 0
            其他五星概率 = 0
        elif 起始已获取UP五星干员乙数量 > 0 and 起始已获取UP五星干员丙数量 == 0:
            乙概率 = 0
            丙概率 = 五星概率
            其他五星概率 = 0
        else:
            乙概率 = 五星概率 / 4
            丙概率 = 五星概率 / 4
            其他五星概率 = 五星概率 / 2

        # 抽到UP6星干员
        转移概率列表.append((获取状态(六星水位=0, 五星水位=0, 已获取UP六星干员数量=起始已获取UP六星干员数量 + 1, 已获取UP五星干员乙数量=起始已获取UP五星干员乙数量, 已获取UP五星干员丙数量=起始已获取UP五星干员丙数量), 六星概率 / 2))

        # 抽到其他6星干员
        转移概率列表.append((获取状态(六星水位=0, 五星水位=0, 已获取UP六星干员数量=起始已获取UP六星干员数量, 已获取UP五星干员乙数量=起始已获取UP五星干员乙数量, 已获取UP五星干员丙数量=起始已获取UP五星干员丙数量), 六星概率 / 2))

        # 抽到UP五星干员乙
        转移概率列表.append((获取状态(六星水位=起始六星水位 + 1, 五星水位=0, 已获取UP六星干员数量=起始已获取UP六星干员数量, 已获取UP五星干员乙数量=起始已获取UP五星干员乙数量 + 1, 已获取UP五星干员丙数量=起始已获取UP五星干员丙数量), 乙概率))

        # 抽到UP五星干员丙
        转移概率列表.append((获取状态(六星水位=起始六星水位 + 1, 五星水位=0, 已获取UP六星干员数量=起始已获取UP六星干员数量, 已获取UP五星干员乙数量=起始已获取UP五星干员乙数量, 已获取UP五星干员丙数量=起始已获取UP五星干员丙数量 + 1), 丙概率))

        # 抽到其他五星干员
        转移概率列表.append((获取状态(六星水位=起始六星水位 + 1, 五星水位=0, 已获取UP六星干员数量=起始已获取UP六星干员数量, 已获取UP五星干员乙数量=起始已获取UP五星干员乙数量, 已获取UP五星干员丙数量=起始已获取UP五星干员丙数量), 其他五星概率))

        # 抽到四星及以下干员
        转移概率列表.append((获取状态(六星水位=起始六星水位 + 1, 五星水位=起始五星水位 + 1, 已获取UP六星干员数量=起始已获取UP六星干员数量, 已获取UP五星干员乙数量=起始已获取UP五星干员乙数量, 已获取UP五星干员丙数量=起始已获取UP五星干员丙数量), 四星或三星概率))

    转移概率列表 = [(目标状态, 概率) for 目标状态, 概率 in 转移概率列表 if 概率 > 0]
    assert np.isclose(sum(x[1] for x in 转移概率列表), 1)
    return 转移概率列表


def 构造状态转移矩阵(状态转移: Callable[[状态类], Iterable[tuple[状态类, float]]]) -> scipy.sparse.csr_array:
    状态数量 = len(状态列表)
    row = []
    col = []
    data = []
    for 起始状态序号, 起始状态 in tqdm.notebook.tqdm(list(enumerate(状态列表))):
        转移概率列表 = 状态转移(起始状态)
        for 目标状态, 概率 in 转移概率列表:
            目标状态序号 = 状态索引[目标状态]
            row.append(起始状态序号)
            col.append(目标状态序号)
            data.append(概率)
    return scipy.sparse.csr_array((data, (row, col)), shape=(状态数量, 状态数量))


已获取UP六星干员数量上限 = 6
已获取UP五星干员乙数量上限 = 6
已获取UP五星干员丙数量上限 = 6


状态列表: list[状态类] = []
状态列表.extend(状态类(六星水位=六星水位, 五星水位=五星水位, 已获取UP六星干员数量=已获取UP六星干员数量, 已获取UP五星干员乙数量=已获取UP五星干员乙数量, 已获取UP五星干员丙数量=已获取UP五星干员丙数量)
            for 六星水位 in range(0, 99, 1)
            for 五星水位 in range(0, 40, 1)
            for 已获取UP六星干员数量 in range(0, 已获取UP六星干员数量上限 + 1, 1)
            for 已获取UP五星干员乙数量 in range(0, 已获取UP五星干员乙数量上限 + 1, 1)
            for 已获取UP五星干员丙数量 in range(0, 已获取UP五星干员丙数量上限 + 1, 1))
状态数量: int = len(状态列表)
状态索引: dict[状态类, int] = {状态: i for i, 状态 in enumerate(状态列表)}

In [ ]:
状态转移矩阵_非第10抽且非第120抽 = 构造状态转移矩阵(partial(状态转移, 是第10抽=False, 是第120抽=False))
状态转移矩阵_第10抽 = 构造状态转移矩阵(partial(状态转移, 是第10抽=True, 是第120抽=False))
状态转移矩阵_第120抽 = 构造状态转移矩阵(partial(状态转移, 是第10抽=False, 是第120抽=True))

scipy.sparse.save_npz("联动寻访中间结果/状态转移矩阵_非第10抽且非第120抽.npz", 状态转移矩阵_非第10抽且非第120抽)
scipy.sparse.save_npz("联动寻访中间结果/状态转移矩阵_第10抽.npz", 状态转移矩阵_第10抽)
scipy.sparse.save_npz("联动寻访中间结果/状态转移矩阵_第120抽.npz", 状态转移矩阵_第120抽)

In [ ]:
状态转移矩阵_非第10抽且非第120抽 = scipy.sparse.load_npz("联动寻访中间结果/状态转移矩阵_非第10抽且非第120抽.npz")
状态转移矩阵_第10抽 = scipy.sparse.load_npz("联动寻访中间结果/状态转移矩阵_第10抽.npz")
状态转移矩阵_第120抽 = scipy.sparse.load_npz("联动寻访中间结果/状态转移矩阵_第120抽.npz")

In [ ]:
迭代次数 = 4096

初始状态 = 状态类(六星水位=0, 五星水位=0, 已获取UP六星干员数量=0, 已获取UP五星干员乙数量=0, 已获取UP五星干员丙数量=0)
当前状态分布 = np.zeros(状态数量)
当前状态分布[状态索引[初始状态]] = 1

# 由于状态数量过多，不在内存中保存历史状态分布，而改为保存 i 抽后已获取甲、乙、丙数量的联合分布

# 历史状态分布 = np.zeros((迭代次数 + 1, 状态数量))
# 历史状态分布[0] = 当前状态分布

# 获得特定数量的状态序号列表数组 = np.empty((已获取UP六星干员数量上限 + 1, 已获取UP五星干员乙数量上限 + 1, 已获取UP五星干员丙数量上限 + 1), dtype=object)
# for a in range(已获取UP六星干员数量上限 + 1):
#     for b in range(已获取UP五星干员乙数量上限 + 1):
#         for c in range(已获取UP五星干员丙数量上限 + 1):
#             获得特定数量的状态序号列表数组[a, b, c] = []
# for 状态序号, 状态 in enumerate(状态列表):
#     获得特定数量的状态序号列表数组[状态.已获取UP六星干员数量, 状态.已获取UP五星干员乙数量, 状态.已获取UP五星干员丙数量].append(状态序号)

历史获得甲乙丙数量联合分布 = np.zeros((迭代次数 + 1, 已获取UP六星干员数量上限 + 1, 已获取UP五星干员乙数量上限 + 1, 已获取UP五星干员丙数量上限 + 1))
"""`历史获得甲乙丙数量联合分布[n, i, j, k]` 表示 n 次寻访后，获取的 (甲, 乙, 丙) 干员数量恰好为 (i, j, k) 的概率"""

状态已获取UP六星干员数量数组 = np.fromiter((s.已获取UP六星干员数量 for s in 状态列表), dtype=int)
状态已获取UP五星干员乙数量数组 = np.fromiter((s.已获取UP五星干员乙数量 for s in 状态列表), dtype=int)
状态已获取UP五星干员丙数量数组 = np.fromiter((s.已获取UP五星干员丙数量 for s in 状态列表), dtype=int)

for i in tqdm.notebook.tqdm(range(迭代次数)):
    if i == 9:
        状态转移矩阵 = 状态转移矩阵_第10抽
    elif i == 119:
        状态转移矩阵 = 状态转移矩阵_第120抽
    else:
        状态转移矩阵 = 状态转移矩阵_非第10抽且非第120抽
    当前状态分布 = 当前状态分布 @ 状态转移矩阵

    # 历史状态分布[i + 1] = 当前状态分布

    当前获得甲乙丙数量联合分布 = np.zeros((已获取UP六星干员数量上限 + 1, 已获取UP五星干员乙数量上限 + 1, 已获取UP五星干员丙数量上限 + 1))
    # for a in range(已获取UP六星干员数量上限 + 1):
    #     for b in range(已获取UP五星干员乙数量上限 + 1):
    #         for c in range(已获取UP五星干员丙数量上限 + 1):
    #             当前获得甲乙丙数量联合分布[a, b, c] = np.sum(当前状态分布[获得特定数量的状态序号列表数组[a, b, c]])
    np.add.at(当前获得甲乙丙数量联合分布, (状态已获取UP六星干员数量数组, 状态已获取UP五星干员乙数量数组, 状态已获取UP五星干员丙数量数组), 当前状态分布)
    # print(当前获得甲乙丙数量联合分布)
    历史获得甲乙丙数量联合分布[i + 1] = 当前获得甲乙丙数量联合分布

np.savez_compressed("联动寻访中间结果/历史获得甲乙丙数量联合分布.npz", 历史获得甲乙丙数量联合分布=历史获得甲乙丙数量联合分布)

In [ ]:
历史获得甲乙丙数量联合分布 = np.load("联动寻访中间结果/历史获得甲乙丙数量联合分布.npz")["历史获得甲乙丙数量联合分布"]

dists: list[FiniteDist] = []
for n in range(已获取UP六星干员数量上限 + 1):
    cdf = np.sum(历史获得甲乙丙数量联合分布[:, n:, :, :], axis=(1, 2, 3))
    dist = FiniteDist(cdf=cdf)
    dists.append(dist)

dists[1] = FiniteDist(dists[1].pk[:120 + 1])

draw_multi_pmf_cdf_fig(dists=dists[1:],
                       labels=[f"{n} 名" for n in range(1, 已获取UP六星干员数量上限 + 1)],
                       title="在联动寻访中获得若干名 UP 6★ 干员所需寻访次数的分布",
                       x_max="auto",
                       save=True)

draw_pmf_cdf_fig(dist=dists[1],
                 title="在联动寻访中获得 1 名 UP 6★ 干员所需寻访次数的分布",
                 x_max=None,
                 save=True)

In [ ]:
历史获得甲乙丙数量联合分布 = np.load("联动寻访中间结果/历史获得甲乙丙数量联合分布.npz")["历史获得甲乙丙数量联合分布"]

数量上限 = min(已获取UP五星干员乙数量上限, 已获取UP五星干员丙数量上限)

dists: list[FiniteDist] = []
for n in range(数量上限 + 1):
    已获取UP五星干员乙数量矩阵 = np.arange(已获取UP五星干员乙数量上限 + 1)[:, None]
    已获取UP五星干员丙数量矩阵 = np.arange(已获取UP五星干员丙数量上限 + 1)[None, :]
    mask = 已获取UP五星干员乙数量矩阵 + 已获取UP五星干员丙数量矩阵 >= n
    cdf = np.sum(历史获得甲乙丙数量联合分布[:, :, mask], axis=(1, 2))
    dist = FiniteDist(cdf=cdf)
    dists.append(dist)

draw_multi_pmf_cdf_fig(dists=dists[1:],
                       labels=[f"{n} 名" for n in range(1, 数量上限 + 1)],
                       title="在联动寻访中获得若干名任意 UP 5★ 干员所需寻访次数的分布",
                       x_max="auto",
                       save=True)

draw_pmf_cdf_fig(dist=dists[1],
                 title="在联动寻访中获得 1 名任意 UP 5★ 干员所需寻访次数的分布",
                 x_max="auto",
                 save=True)

In [ ]:
历史获得甲乙丙数量联合分布 = np.load("联动寻访中间结果/历史获得甲乙丙数量联合分布.npz")["历史获得甲乙丙数量联合分布"]

dists: list[FiniteDist] = []
for n in range(已获取UP五星干员乙数量上限 + 1):
    cdf = np.sum(历史获得甲乙丙数量联合分布[:, :, n:, :], axis=(1, 2, 3))
    dist = FiniteDist(cdf=cdf)
    dists.append(dist)

draw_multi_pmf_cdf_fig(dists=dists[1:],
                       labels=[f"{n} 名" for n in range(1, 已获取UP五星干员乙数量上限 + 1)],
                       title="在联动寻访中获得若干名指定的 UP 5★ 干员所需寻访次数的分布",
                       x_max="auto",
                       save=True)

draw_pmf_cdf_fig(dist=dists[1],
                 title="在联动寻访中获得 1 名指定的 UP 5★ 干员所需寻访次数的分布",
                 x_max="auto",
                 save=True)

In [ ]:
历史获得甲乙丙数量联合分布 = np.load("联动寻访中间结果/历史获得甲乙丙数量联合分布.npz")["历史获得甲乙丙数量联合分布"]

dists = [FiniteDist([1])]

数量上限 = min(已获取UP六星干员数量上限, 已获取UP五星干员乙数量上限, 已获取UP五星干员丙数量上限)

for n in range(1, 数量上限 + 1):
    cdf = np.sum(历史获得甲乙丙数量联合分布[:, :, n:, n:], axis=(1, 2, 3))
    dist = FiniteDist(cdf=cdf)
    dists.append(dist)

draw_multi_pmf_cdf_fig(dists=dists[1:],
                       labels=[f"各至少 {n} 次" for n in range(1, 数量上限 + 1)],
                       title="在联动寻访中获得全部 2 名 UP 5★ 干员各至少若干次所需寻访次数的分布",
                       x_max="auto",
                       save=True)

draw_pmf_cdf_fig(dist=dists[1],
                 title="在联动寻访中获得全部 2 名 UP 5★ 干员各至少 1 次所需寻访次数的分布",
                 x_max="auto",
                 save=True)


In [ ]:
历史获得甲乙丙数量联合分布 = np.load("联动寻访中间结果/历史获得甲乙丙数量联合分布.npz")["历史获得甲乙丙数量联合分布"]

dists = [FiniteDist([1])]

数量上限 = min(已获取UP六星干员数量上限, 已获取UP五星干员乙数量上限, 已获取UP五星干员丙数量上限)

for n in range(1, 数量上限 + 1):
    cdf = np.sum(历史获得甲乙丙数量联合分布[:, n:, n:, n:], axis=(1, 2, 3))
    dist = FiniteDist(cdf=cdf)
    dists.append(dist)

draw_multi_pmf_cdf_fig(dists=dists[1:],
                       labels=[f"各至少 {n} 次" for n in range(1, 数量上限 + 1)],
                       title="在联动寻访中获得 UP 6★ 干员和全部 2 名 UP 5★ 干员各至少若干次所需寻访次数的分布",
                       x_max="auto",
                       save=True)

draw_pmf_cdf_fig(dist=dists[1],
                 title="在联动寻访中获得 UP 6★ 干员和全部 2 名 UP 5★ 干员各至少 1 次所需寻访次数的分布",
                 x_max="auto",
                 save=True)
